# Notebook 19 — Merged Multi-Dataset Balanced Training

## Strategy: Add IDRiD + Messidor Grades 1,2,3,4 → Balance All Classes

### Current Problem (APTOS-only training):
| Grade | Count | Ratio |
|-------|-------|-------|
| G0 | 1,258 | 1:1 |
| G1 | 228 | 1:5.5 |
| G2 | 665 | 1:1.9 |
| G3 | **127** | **1:9.9** |
| G4 | **195** | **1:6.5** |

### Solution:
1. Keep ALL APTOS training data (all grades)
2. Add IDRiD + Messidor **grades 1, 2, 3, 4 only** (skip G0 — already have too many)
3. Apply **WeightedRandomSampler** to balance all 5 classes equally per epoch
4. Use **class-weighted loss** + **Mixup** for extra regularization
5. **Val/Test remain APTOS-only** for fair comparison with previous models

### After Merging:
| Grade | APTOS | +IDRiD | +Messidor | Total |
|-------|-------|--------|-----------|-------|
| G0 | 1,258 | — | — | 1,258 |
| G1 | 228 | +50 | +187 | 465 |
| G2 | 665 | +355 | +226 | 1,246 |
| G3 | 127 | +186 | +56 | 369 |
| G4 | 195 | +139 | +22 | 356 |

In [ ]:
# ============================================================
# 1. Imports & Configuration
# ============================================================
import os
import gc
import copy
import json
import time
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report

warnings.filterwarnings("ignore")

# ── Reproducibility ──
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.use_deterministic_algorithms(False)

set_seed(SEED)

# ── Device ──
def get_device():
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()

# ── Configuration ──
CONFIG = {
    "seed": SEED,
    "device": str(DEVICE),
    "data_root": ".",
    "models_dir": "models",
    "figures_dir": "figures",
    # Model
    "model_name": "efficientnet_b3",
    "pretrained": True,
    "num_classes": 5,
    "dropout_rate": 0.4,
    # Image
    "image_size": 384,
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
    # Training — Phase 1
    "phase1_epochs": 5,
    "phase1_lr": 1e-3,
    # Training — Phase 2
    "phase2_epochs": 40,
    "phase2_lr": 1e-4,
    "weight_decay": 1e-4,
    # Scheduler
    "scheduler_T0": 10,
    "scheduler_Tmult": 2,
    # DataLoader
    "batch_size": 16,
    "num_workers": 0,
    "pin_memory": False,
    # Regularisation
    "grad_clip_max_norm": 1.0,
    "label_smoothing": 0.1,
    # Mixup
    "mixup_alpha": 0.2,
    "mixup_prob": 0.3,
    # Early stopping
    "early_stopping_patience": 12,
    # Mixed precision
    "use_amp": DEVICE.type in ("cuda",),
    # Balancing
    "use_weighted_sampler": True,
    "use_class_weights": True,
}

ROOT = Path(CONFIG["data_root"])
MODELS_DIR = ROOT / CONFIG["models_dir"]
FIG_DIR = ROOT / CONFIG["figures_dir"]
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device       : {DEVICE}")
print(f"Model        : {CONFIG['model_name']}")
print(f"Phase 1      : {CONFIG['phase1_epochs']} epochs, lr={CONFIG['phase1_lr']}")
print(f"Phase 2      : {CONFIG['phase2_epochs']} epochs, lr={CONFIG['phase2_lr']}")
print(f"Mixup        : alpha={CONFIG['mixup_alpha']}, prob={CONFIG['mixup_prob']}")
print(f"Early stop   : patience={CONFIG['early_stopping_patience']}")
print("Configuration loaded ✓")

## 2. Build Merged Training Set

Strategy:
- Keep **ALL** APTOS training data (all 5 grades)
- Add IDRiD + Messidor **grades 1, 2, 3, 4 only**
- Skip IDRiD/Messidor Grade 0 (already have 1,258 from APTOS)
- Val/Test stay **APTOS-only** for fair comparison

In [ ]:
# ============================================================
# 2. Build Merged Training Dataset
# ============================================================

# Load APTOS training split (original)
df_aptos_train = pd.read_csv('splits_aptos/train_split.csv')
print(f"APTOS train: {len(df_aptos_train)} images")

# Load APTOS val (unchanged)
df_val = pd.read_csv('splits_aptos/val_split.csv')
print(f"APTOS val:   {len(df_val)} images")

# ── Load IDRiD ──
# Use the merged splits which already have preprocessed_path mapping
df_merged_train = pd.read_csv('splits/train_split.csv')

# Extract IDRiD grades 1,2,3,4 from merged train
df_idrid_extra = df_merged_train[
    (df_merged_train['original_dataset'] == 'idrid') &
    (df_merged_train['dr_grade'] >= 1)
].copy()
print(f"\nIDRiD (grades 1-4): {len(df_idrid_extra)} images")
print(df_idrid_extra['dr_grade'].value_counts().sort_index().to_string())

# Extract Messidor grades 1,2,3,4 from merged train
df_messidor_extra = df_merged_train[
    (df_merged_train['original_dataset'] == 'messidor2') &
    (df_merged_train['dr_grade'] >= 1)
].copy()
print(f"\nMessidor (grades 1-4): {len(df_messidor_extra)} images")
print(df_messidor_extra['dr_grade'].value_counts().sort_index().to_string())

# ── Merge: APTOS all + IDRiD G1-4 + Messidor G1-4 ──
# Ensure consistent columns
keep_cols = ['image_id', 'original_dataset', 'dr_grade', 'binary_label', 
             'file_path', 'preprocessed_path']

# APTOS might not have 'original_dataset', add it
df_aptos_train = df_aptos_train.copy()
if 'original_dataset' not in df_aptos_train.columns:
    df_aptos_train['original_dataset'] = 'aptos2019'
if 'binary_label' not in df_aptos_train.columns:
    df_aptos_train['binary_label'] = (df_aptos_train['dr_grade'] >= 1).astype(int)

# Select common columns that exist in both
common_cols = [c for c in keep_cols if c in df_aptos_train.columns and c in df_idrid_extra.columns]

df_train_merged = pd.concat([
    df_aptos_train[common_cols],
    df_idrid_extra[common_cols],
    df_messidor_extra[common_cols],
], ignore_index=True)

print(f"\n{'='*60}")
print(f"MERGED TRAINING SET: {len(df_train_merged)} images")
print(f"{'='*60}")
print(f"\n{'Grade':<10} {'APTOS':>7} {'IDRiD':>7} {'Messidor':>8} {'Total':>7} {'Pct':>6}")
print("-" * 55)

for g in range(5):
    a = len(df_aptos_train[df_aptos_train['dr_grade'] == g])
    i = len(df_idrid_extra[df_idrid_extra['dr_grade'] == g])
    m = len(df_messidor_extra[df_messidor_extra['dr_grade'] == g])
    t = len(df_train_merged[df_train_merged['dr_grade'] == g])
    pct = t / len(df_train_merged) * 100
    print(f"Grade {g:<4} {a:>7} {i:>7} {m:>8} {t:>7} {pct:>5.1f}%")
print(f"{'Total':<10} {len(df_aptos_train):>7} {len(df_idrid_extra):>7} {len(df_messidor_extra):>8} {len(df_train_merged):>7}")

# Verify preprocessed images exist
missing = 0
for _, row in df_train_merged.iterrows():
    if not Path(row['preprocessed_path']).exists():
        missing += 1
print(f"\nMissing preprocessed images: {missing}")
if missing > 0:
    # Remove missing
    df_train_merged = df_train_merged[df_train_merged['preprocessed_path'].apply(lambda p: Path(p).exists())]
    print(f"After removing missing: {len(df_train_merged)} images")

## 3. Visualize Before vs After Balancing

In [ ]:
# ============================================================
# 3. Visualize Distribution
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

grade_names = ['G0\nNo DR', 'G1\nMild', 'G2\nModerate', 'G3\nSevere', 'G4\nPDR']
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']

# APTOS only
ax = axes[0]
aptos_counts = [len(df_aptos_train[df_aptos_train['dr_grade'] == g]) for g in range(5)]
bars = ax.bar(range(5), aptos_counts, color=colors, alpha=0.8)
ax.set_xticks(range(5))
ax.set_xticklabels(grade_names)
ax.set_title('APTOS Only (Original)', fontsize=13)
ax.set_ylabel('Count')
for bar, c in zip(bars, aptos_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(c),
            ha='center', fontsize=10)

# Merged (before sampler)
ax = axes[1]
merged_counts = [len(df_train_merged[df_train_merged['dr_grade'] == g]) for g in range(5)]
bars = ax.bar(range(5), merged_counts, color=colors, alpha=0.8)
ax.set_xticks(range(5))
ax.set_xticklabels(grade_names)
ax.set_title('After Merging IDRiD+Messidor G1-4', fontsize=13)
ax.set_ylabel('Count')
for bar, c in zip(bars, merged_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(c),
            ha='center', fontsize=10)

# After balanced sampler (expected)
ax = axes[2]
target_per_class = len(df_train_merged) // 5
balanced_counts = [target_per_class] * 5
bars = ax.bar(range(5), balanced_counts, color=colors, alpha=0.8)
ax.set_xticks(range(5))
ax.set_xticklabels(grade_names)
ax.set_title('After WeightedRandomSampler', fontsize=13)
ax.set_ylabel('Expected Samples/Epoch')
for bar, c in zip(bars, balanced_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, f'~{c}',
            ha='center', fontsize=10)

plt.suptitle('Class Distribution: Before → Merge → Balance', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'merged_balanced_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Dataset & Augmentation

In [ ]:
# ============================================================
# 4. Dataset & Augmentation
# ============================================================

def get_train_transforms(image_size, mean, std):
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.1, scale_limit=0.15, rotate_limit=30,
            border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5
        ),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20),
            A.RandomGamma(gamma_limit=(80, 120)),
        ], p=0.4),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5)),
            A.GaussNoise(std_range=(0.02, 0.05)),
        ], p=0.2),
        A.CoarseDropout(
            num_holes_range=(1, 3),
            hole_height_range=(int(image_size * 0.05), int(image_size * 0.15)),
            hole_width_range=(int(image_size * 0.05), int(image_size * 0.15)),
            fill="random", p=0.3,
        ),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])

def get_val_transforms(image_size, mean, std):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])


class DRDataset(Dataset):
    def __init__(self, dataframe, transform, data_root='.'):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.data_root = Path(data_root)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_root / row['preprocessed_path']
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f"Cannot read: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = self.transform(image=img)
        image = augmented['image']
        label = torch.tensor(row['dr_grade'], dtype=torch.long)
        return image, label


print("Dataset and transforms defined ✓")

## 5. Create Balanced DataLoaders

In [ ]:
# ============================================================
# 5. Balanced DataLoaders
# ============================================================

train_tfm = get_train_transforms(CONFIG['image_size'], CONFIG['mean'], CONFIG['std'])
val_tfm = get_val_transforms(CONFIG['image_size'], CONFIG['mean'], CONFIG['std'])

train_ds = DRDataset(df_train_merged, train_tfm)
val_ds = DRDataset(df_val, val_tfm)

# ── WeightedRandomSampler: FULL balance (power=1.0) ──
# Every class sees equal samples per epoch
labels = df_train_merged['dr_grade'].values
class_counts = np.bincount(labels, minlength=5).astype(np.float64)

# Inverse frequency → each class equally likely
class_sample_weights = 1.0 / class_counts
class_sample_weights = class_sample_weights / class_sample_weights.sum()

sample_weights = np.array([class_sample_weights[label] for label in labels])
sample_weights = torch.from_numpy(sample_weights).double()

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_ds),
    replacement=True,
)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG['batch_size'],
    sampler=sampler,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory'],
    drop_last=True,
)

val_loader = DataLoader(
    val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'],
)

# Show expected distribution
expected_per_class = class_sample_weights * len(train_ds)
print(f"{'Grade':<10} {'Actual':>8} {'Expected/Epoch':>15} {'Oversample':>12}")
print("-" * 50)
for g in range(5):
    actual = int(class_counts[g])
    expected = expected_per_class[g]
    ratio = expected / actual
    print(f"Grade {g:<4} {actual:>8} {expected:>15.0f} {ratio:>11.2f}x")

print(f"\nTrain: {len(train_ds)} images → {len(train_loader)} batches/epoch")
print(f"Val:   {len(val_ds)} images → {len(val_loader)} batches")
print(f"\nFull balance: each class sees ~{len(train_ds)//5} samples per epoch")

# ── Class weights for loss ──
class_weights_loss = 1.0 / (class_counts + 1e-6)
class_weights_loss = class_weights_loss / class_weights_loss.sum() * CONFIG['num_classes']
class_weights_tensor = torch.tensor(class_weights_loss, dtype=torch.float32).to(DEVICE)
print(f"\nLoss class weights: {np.round(class_weights_loss, 3)}")

## 6. Model Architecture

In [ ]:
# ============================================================
# 6. Model (same architecture as Notebook 04)
# ============================================================

def create_multiclass_model(model_name='efficientnet_b3', pretrained=True,
                             num_classes=5, dropout_rate=0.4):
    model = timm.create_model(model_name, pretrained=pretrained)
    n_features = model.num_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(n_features, num_classes),
    )
    return model, n_features

model, n_features = create_multiclass_model(
    CONFIG['model_name'], CONFIG['pretrained'],
    CONFIG['num_classes'], CONFIG['dropout_rate'],
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {CONFIG['model_name']}")
print(f"  Head: Dropout({CONFIG['dropout_rate']}) → Linear({n_features}, {CONFIG['num_classes']})")
print(f"  Total params: {total_params:,}")

## 7. Training Utilities

In [ ]:
# ============================================================
# 7. Training Utilities
# ============================================================

def quadratic_weighted_kappa(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode='max'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.should_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False
        improved = (score > self.best_score + self.min_delta) if self.mode == 'max' \
                   else (score < self.best_score - self.min_delta)
        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                return True
        return False

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.avg = 0.0; self.sum = 0.0; self.count = 0
    def update(self, val, n=1):
        self.sum += val * n; self.count += n; self.avg = self.sum / self.count

def freeze_backbone(model):
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Backbone FROZEN — trainable: {trainable:,}")

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  All UNFROZEN — trainable: {trainable:,}")

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("Utilities defined ✓")

## 8. Training & Validation Functions

In [ ]:
# ============================================================
# 8. Train/Val functions
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device,
                    grad_clip=1.0, use_amp=False, scaler=None,
                    mixup_alpha=0.0, mixup_prob=0.0):
    model.train()
    loss_meter = AverageMeter()
    all_labels, all_preds = [], []

    pbar = tqdm(loader, desc='  Train', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()

        use_mixup = mixup_alpha > 0 and random.random() < mixup_prob

        if use_amp and scaler is not None:
            with torch.amp.autocast('cuda'):
                if use_mixup:
                    mixed, ya, yb, lam = mixup_data(images, labels, mixup_alpha)
                    logits = model(mixed)
                    loss = mixup_criterion(criterion, logits, ya, yb, lam)
                else:
                    logits = model(images)
                    loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            if use_mixup:
                mixed, ya, yb, lam = mixup_data(images, labels, mixup_alpha)
                logits = model(mixed)
                loss = mixup_criterion(criterion, logits, ya, yb, lam)
            else:
                logits = model(images)
                loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        loss_meter.update(loss.item(), images.size(0))

        with torch.no_grad():
            if use_mixup:
                preds = torch.argmax(model(images), dim=1).cpu().numpy()
            else:
                preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f'{loss_meter.avg:.4f}')

    acc = accuracy_score(all_labels, all_preds)
    qwk = quadratic_weighted_kappa(np.array(all_labels), np.array(all_preds))
    return loss_meter.avg, acc, qwk


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    loss_meter = AverageMeter()
    all_labels, all_preds = [], []

    pbar = tqdm(loader, desc='  Val  ', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss_meter.update(loss.item(), images.size(0))
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f'{loss_meter.avg:.4f}')

    acc = accuracy_score(all_labels, all_preds)
    qwk = quadratic_weighted_kappa(np.array(all_labels), np.array(all_preds))
    return loss_meter.avg, acc, qwk

print("Training functions defined ✓")

## 9. Phase 1 — Frozen Backbone

In [ ]:
# ============================================================
# 9. Phase 1 — Frozen backbone
# ============================================================
print("=" * 60)
print("PHASE 1: Frozen backbone — train head only")
print(f"Training on {len(df_train_merged)} images (APTOS + IDRiD G1-4 + Messidor G1-4)")
print(f"Validating on {len(df_val)} images (APTOS only)")
print("=" * 60)

freeze_backbone(model)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=CONFIG['label_smoothing'],
)

optimizer_p1 = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['phase1_lr'],
    weight_decay=CONFIG['weight_decay'],
)

scaler = torch.amp.GradScaler('cuda') if CONFIG['use_amp'] else None

history = {
    'epoch': [], 'phase': [],
    'train_loss': [], 'train_acc': [], 'train_qwk': [],
    'val_loss': [], 'val_acc': [], 'val_qwk': [],
    'lr': [],
}

best_val_qwk = 0.0
best_model_state = None

print(f"\n{'Epoch':>6} {'Phase':>7} {'TrLoss':>8} {'TrAcc':>7} {'TrQWK':>7} "
      f"{'VlLoss':>8} {'VlAcc':>7} {'VlQWK':>7} {'LR':>10} {'Best':>5}")
print("-" * 85)

for epoch in range(1, CONFIG['phase1_epochs'] + 1):
    train_loss, train_acc, train_qwk = train_one_epoch(
        model, train_loader, criterion, optimizer_p1, DEVICE,
        grad_clip=CONFIG['grad_clip_max_norm'],
        use_amp=CONFIG['use_amp'], scaler=scaler,
        mixup_alpha=0, mixup_prob=0,
    )
    val_loss, val_acc, val_qwk = validate(model, val_loader, criterion, DEVICE)

    lr = optimizer_p1.param_groups[0]['lr']
    is_best = val_qwk > best_val_qwk
    if is_best:
        best_val_qwk = val_qwk
        best_model_state = copy.deepcopy(model.state_dict())

    history['epoch'].append(epoch)
    history['phase'].append('phase1')
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_qwk'].append(train_qwk)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)
    history['lr'].append(lr)

    marker = '  ★' if is_best else ''
    print(f"{epoch:6d} {'P1':>7} {train_loss:8.4f} {train_acc:7.4f} {train_qwk:7.4f} "
          f"{val_loss:8.4f} {val_acc:7.4f} {val_qwk:7.4f} {lr:10.6f}{marker}")

print(f"\nPhase 1 done — Best val QWK: {best_val_qwk:.4f}")

## 10. Phase 2 — Full Fine-Tuning with Mixup

In [ ]:
# ============================================================
# 10. Phase 2 — Full fine-tuning
# ============================================================
print("=" * 60)
print("PHASE 2: Full fine-tuning + Balanced Sampling + Mixup")
print("=" * 60)

unfreeze_all(model)

optimizer_p2 = AdamW(
    [
        {'params': [p for n, p in model.named_parameters() if 'classifier' not in n],
         'lr': CONFIG['phase2_lr'] * 0.1},
        {'params': model.classifier.parameters(),
         'lr': CONFIG['phase2_lr']},
    ],
    weight_decay=CONFIG['weight_decay'],
)
print(f"  Backbone lr: {CONFIG['phase2_lr'] * 0.1:.1e}")
print(f"  Head lr    : {CONFIG['phase2_lr']:.1e}")

scheduler = CosineAnnealingWarmRestarts(
    optimizer_p2, T_0=CONFIG['scheduler_T0'], T_mult=CONFIG['scheduler_Tmult'],
)

early_stopper = EarlyStopping(patience=CONFIG['early_stopping_patience'], mode='max')

epoch_offset = CONFIG['phase1_epochs']
print(f"\n{'Epoch':>6} {'Phase':>7} {'TrLoss':>8} {'TrAcc':>7} {'TrQWK':>7} "
      f"{'VlLoss':>8} {'VlAcc':>7} {'VlQWK':>7} {'LR':>10} {'Best':>5}")
print("-" * 85)

for epoch in range(1, CONFIG['phase2_epochs'] + 1):
    global_epoch = epoch_offset + epoch

    train_loss, train_acc, train_qwk = train_one_epoch(
        model, train_loader, criterion, optimizer_p2, DEVICE,
        grad_clip=CONFIG['grad_clip_max_norm'],
        use_amp=CONFIG['use_amp'], scaler=scaler,
        mixup_alpha=CONFIG['mixup_alpha'],
        mixup_prob=CONFIG['mixup_prob'],
    )
    val_loss, val_acc, val_qwk = validate(model, val_loader, criterion, DEVICE)

    scheduler.step()
    lr = optimizer_p2.param_groups[1]['lr']

    is_best = val_qwk > best_val_qwk
    if is_best:
        best_val_qwk = val_qwk
        best_model_state = copy.deepcopy(model.state_dict())
        checkpoint = {
            'model_name': CONFIG['model_name'],
            'num_features': n_features,
            'dropout_rate': CONFIG['dropout_rate'],
            'num_classes': CONFIG['num_classes'],
            'image_size': CONFIG['image_size'],
            'mean': CONFIG['mean'],
            'std': CONFIG['std'],
            'model_state_dict': best_model_state,
            'best_val_qwk': best_val_qwk,
            'config': CONFIG,
            'training_data': {
                'aptos_train': len(df_aptos_train),
                'idrid_g1_4': len(df_idrid_extra),
                'messidor_g1_4': len(df_messidor_extra),
                'total': len(df_train_merged),
            },
        }
        torch.save(checkpoint, MODELS_DIR / 'best_multiclass_merged_balanced_model.pth')

    history['epoch'].append(global_epoch)
    history['phase'].append('phase2')
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_qwk'].append(train_qwk)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)
    history['lr'].append(lr)

    marker = '  ★' if is_best else ''
    print(f"{global_epoch:6d} {'P2':>7} {train_loss:8.4f} {train_acc:7.4f} {train_qwk:7.4f} "
          f"{val_loss:8.4f} {val_acc:7.4f} {val_qwk:7.4f} {lr:10.6f}{marker}")

    if early_stopper(val_qwk):
        print(f"\n  Early stopping at epoch {global_epoch}")
        break

    if DEVICE.type == 'mps':
        torch.mps.empty_cache()
    gc.collect()

print(f"\nPhase 2 done — Best val QWK: {best_val_qwk:.4f}")

## 11. Save History & Training Curves

In [ ]:
# ============================================================
# 11. Save history
# ============================================================
df_history = pd.DataFrame(history)
df_history.to_csv(MODELS_DIR / 'multiclass_merged_balanced_history.csv', index=False)

# Plot
phase1_end = CONFIG['phase1_epochs']
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle('Merged Balanced Training (APTOS + IDRiD/Messidor G1-4)', fontsize=15, y=1.02)

for ax, metric, title in [
    (axes[0], ('train_loss', 'val_loss'), 'Loss'),
    (axes[1], ('train_acc', 'val_acc'), 'Accuracy'),
    (axes[2], ('train_qwk', 'val_qwk'), 'QWK'),
]:
    ax.plot(df_history['epoch'], df_history[metric[0]], 'b-o', markersize=3, label=f'Train')
    ax.plot(df_history['epoch'], df_history[metric[1]], 'r-s', markersize=3, label=f'Val')
    ax.axvline(x=phase1_end + 0.5, color='gray', linestyle='--', alpha=0.5, label='Phase1→2')
    ax.set_xlabel('Epoch')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'merged_balanced_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {FIG_DIR / 'merged_balanced_training_curves.png'}")

## 12. Validation Report

In [ ]:
# ============================================================
# 12. Validation report
# ============================================================
model.load_state_dict(best_model_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Evaluating'):
        images = images.to(DEVICE)
        logits = model(images)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print('=' * 60)
print('VALIDATION RESULTS — MERGED BALANCED MODEL')
print('=' * 60)
acc = accuracy_score(all_labels, all_preds)
qwk = quadratic_weighted_kappa(all_labels, all_preds)
print(f'Accuracy: {acc:.4f}')
print(f'QWK:      {qwk:.4f}')

grade_names = ['G0: No DR', 'G1: Mild', 'G2: Moderate', 'G3: Severe', 'G4: PDR']
print(f'\n{classification_report(all_labels, all_preds, target_names=grade_names, digits=4, zero_division=0)}')

# Compare with previous models
print('\n' + '=' * 60)
print('COMPARISON WITH PREVIOUS MODELS')
print('=' * 60)
print(f'{"Model":<45} {"Val QWK":>8}')
print('-' * 55)
print(f'{"Original multiclass (Notebook 04)":<45} {"0.8636":>8}')
print(f'{"Merged balanced (this notebook)":<45} {qwk:>8.4f}')
print(f'{"Difference":<45} {qwk - 0.8636:>+8.4f}')

## 13. Summary

### Training Setup:
- **Train**: APTOS all grades + IDRiD grades 1-4 + Messidor grades 1-4 = ~3,700 images
- **Val**: APTOS-only (530 images) — fair comparison
- **Balancing**: Full WeightedRandomSampler → ~740 samples per class per epoch
- **Extra**: Class-weighted loss + Mixup (α=0.2) + Enhanced augmentation
- **Output**: `models/best_multiclass_merged_balanced_model.pth`

### Next Steps:
1. Run **Notebook 17** with the new checkpoint to evaluate on the APTOS test set
2. Compare QWK, per-class accuracy (especially G1, G3, G4) against original model
3. If improved → use this model in the thesis